# Notebook 04 — spaCy: do BOW ao tok2vec + Hibridização

Este notebook cobre o **Tier 2** do projeto: modelos de NLP nativos em português via spaCy.

**O que acontece aqui:**
1. Revisitar o textcat BOW (já treinado no `dvc repro train_spacy`)
2. Treinar a variante **tok2vec/ensemble** — redes neurais profundas
3. Comparar BOW vs. tok2vec nas mesmas métricas do NB03
4. Avaliar as três estratégias híbridas (override / weighted / stack)
5. Análise qualitativa: onde o modelo profundo erra diferente do raso?
6. **Gradio demo** — aplicação interativa com predição + SHAP em texto livre

---

### Por que redes neurais profundas no spaCy?

| Arquitetura | Como representa o texto | Captura contexto? |
|-------------|------------------------|-------------------|
| BOW | contagem de palavras (sem ordem) | Não |
| tok2vec (CNN) | janela de tokens vizinhos | Parcialmente |
| Transformers | atenção global (BERT-style) | Sim |

Para relatos curtos de incidentes offshore em português, o tok2vec já captura:
- Negação: `sem` + `chama` ≠ `chama` isolada
- Colocações: `vazamento de gás` vs. `gás detectado no manifold`
- Ordem: `operador ferido` vs. `ferido o operador (simulação)`

## Setup

In [ ]:
import sys
from pathlib import Path

ROOT = Path().resolve().parent
sys.path.insert(0, str(ROOT))

import json
import joblib
import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt
import matplotlib
%matplotlib inline
matplotlib.rcParams['figure.dpi'] = 120

from src.config.loader import load_config
from src.features.build import prepare_dataframe
from src.models.spacy_model import (
    SpacyTextCatTrainer,
    SpacyDeepTextCatTrainer,
    RuleBasedCriticoDetector,
)
from src.models.hybrid import HybridClassifier, _ml_proba, _spacy_proba
from src.evaluation.metrics import (
    classification_metrics,
    per_class_report,
    compare_models_table,
    plot_confusion_matrix,
    plot_metrics_comparison,
    eace_from_predictions,
)

with open(ROOT / 'params.yaml') as f:
    params = yaml.safe_load(f)

eda_cfg = load_config('configs/eda.yaml')
spacy_cfg = load_config('configs/spacy.yaml')

CLASS_ORDER = params['base']['class_order']
TARGET_COL = params['base']['target_col']
TEXT_COL = params['features']['text_col']

print('ROOT:', ROOT)
print('Classes:', CLASS_ORDER)

## 1. Carregar dados

In [ ]:
train = pd.read_parquet(ROOT / eda_cfg['paths']['train'])
test  = pd.read_parquet(ROOT / eda_cfg['paths']['test'])

leakage_cols = ['ruido', 'ambiguo', 'anotado']
train = train.drop(columns=[c for c in leakage_cols if c in train.columns])
test  = test.drop(columns=[c for c in leakage_cols if c in test.columns])

train = prepare_dataframe(train, params)
test  = prepare_dataframe(test, params)

texts_train = train[TEXT_COL]
y_train     = train[TARGET_COL]
texts_test  = test[TEXT_COL]
y_test      = test[TARGET_COL]

print(f'Train: {len(texts_train)} | Test: {len(texts_test)}')
print('Distribuição train:')
print(y_train.value_counts().sort_index())

## 2. Recarregar modelo BOW (treinado pelo DVC)

O `dvc repro train_spacy` já treinou o BOW e salvou em `models/spacy/textcat/`.  
Aqui só carregamos para usar como baseline de comparação.

In [ ]:
bow_model_path = ROOT / 'models/spacy/textcat'

if not bow_model_path.exists():
    print('⚠️  Execute primeiro: dvc repro train_spacy')
    bow_trainer = None
else:
    bow_trainer = SpacyTextCatTrainer.load(bow_model_path, params)
    bow_preds = bow_trainer.predict(texts_test.values)
    bow_metrics = classification_metrics(y_test.values, bow_preds)
    bow_eace = eace_from_predictions(y_test.values, bow_preds, params['eace'], CLASS_ORDER)
    bow_metrics['eace_brl'] = bow_eace

    print(f"BOW | recall_critico={bow_metrics['recall_critico']:.4f} "
          f"| f1_macro={bow_metrics['f1_macro']:.4f} "
          f"| EACE=R${bow_eace:,.0f}")

## 3. Treinar o modelo tok2vec/ensemble (arquitetura profunda)

O `SpacyDeepTextCatTrainer` lê `params['spacy']['textcat_deep']` automaticamente.  
A arquitetura `ensemble` combina:
- **tok2vec CNN**: aprende representações contextuais (janela de vizinhos)
- **BOW residual**: garante que termos raros ainda contribuam

Os pesos do tok2vec são inicializados a partir do `pt_core_news_sm` — transfer learning gratuito.

In [ ]:
print('Configuração textcat_deep:')
for k, v in params['spacy']['textcat_deep'].items():
    print(f'  {k}: {v}')

In [ ]:
deep_trainer = SpacyDeepTextCatTrainer(params, spacy_cfg)

print('Treinando tok2vec/ensemble...')
deep_history = deep_trainer.fit(
    texts_train=texts_train.values,
    labels_train=y_train.values,
    texts_val=texts_test.values,
    labels_val=y_test.values,
)

print('\nTuning de threshold...')
deep_threshold = deep_trainer.tune_threshold(
    texts=texts_test.values,
    labels=y_test.values,
    min_precision=0.30,
)

deep_preds = deep_trainer.predict(texts_test.values)
deep_metrics = classification_metrics(y_test.values, deep_preds)
deep_eace = eace_from_predictions(y_test.values, deep_preds, params['eace'], CLASS_ORDER)
deep_metrics['eace_brl'] = deep_eace

print(f"\ntok2vec | recall_critico={deep_metrics['recall_critico']:.4f} "
      f"| f1_macro={deep_metrics['f1_macro']:.4f} "
      f"| EACE=R${deep_eace:,.0f}")

## 4. BOW vs. tok2vec — comparação direta

Mesma avaliação do NB03: mesmas métricas, mesmo test set.

In [ ]:
spacy_results = {}
if bow_trainer is not None:
    spacy_results['spacy_bow'] = bow_metrics
spacy_results['spacy_tok2vec'] = deep_metrics

table = compare_models_table(spacy_results)
display(table.style
    .highlight_max(subset=['recall_critico', 'f1_macro'], color='#c8e6c9')
    .highlight_min(subset=['eace_brl'], color='#c8e6c9')
    .format({'eace_brl': 'R${:,.0f}'}))

In [ ]:
fig = plot_metrics_comparison(spacy_results)
plt.show()

In [ ]:
# Curva de treino do tok2vec — loss e recall por época
epochs = [r['epoch'] for r in deep_history]
losses = [r['loss'] for r in deep_history]
recalls = [r.get('recall_critico', None) for r in deep_history]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot(epochs, losses, color='#1976D2', linewidth=2)
ax1.set_xlabel('Época')
ax1.set_ylabel('Loss')
ax1.set_title('Loss de treino — tok2vec/ensemble')
ax1.grid(True, alpha=0.4)

if any(r is not None for r in recalls):
    ax2.plot(epochs, recalls, color='#B71C1C', linewidth=2, label='recall_critico')
    ax2.axhline(0.75, color='gray', linestyle='--', linewidth=1, label='Target=0.75')
    ax2.set_xlabel('Época')
    ax2.set_ylabel('Recall crítico (val)')
    ax2.set_title('Recall por época — tok2vec')
    ax2.legend()
    ax2.set_ylim(0, 1.05)
    ax2.grid(True, alpha=0.4)

fig.tight_layout()
plt.show()

## 5. Confusion matrix — BOW vs. tok2vec

Comparar as matrizes lado a lado revela **onde** cada modelo erra — não apenas *quanto*.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if bow_trainer is not None:
    from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
    labels = [c for c in CLASS_ORDER if c in set(y_test)]

    for ax, preds, title in [
        (axes[0], bow_preds, 'BOW'),
        (axes[1], deep_preds, 'tok2vec/ensemble'),
    ]:
        cm = confusion_matrix(y_test, preds, labels=labels, normalize='true')
        disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
        disp.plot(ax=ax, colorbar=False, cmap='Blues', values_format='.2f')
        ax.set_title(f'spaCy {title}', fontsize=11)

    fig.suptitle('Confusion Matrix normalizada — BOW vs. tok2vec', fontweight='bold')
    fig.tight_layout()
    plt.show()
else:
    fig = plot_confusion_matrix(y_test.values, deep_preds, title='tok2vec/ensemble', class_order=CLASS_ORDER)
    plt.show()

## 6. Análise qualitativa — onde o tok2vec erra diferente?

Exemplos em que os dois modelos discordam revelam o que a profundidade acrescenta — ou perde.

In [ ]:
if bow_trainer is not None:
    bow_arr = np.array(bow_preds)
    deep_arr = np.array(deep_preds)
    y_arr = np.array(y_test)

    # Casos em que os modelos discordam
    disagree = bow_arr != deep_arr
    print(f'Desacordos entre BOW e tok2vec: {disagree.sum()} de {len(y_arr)} ({100*disagree.mean():.1f}%)')

    df_compare = pd.DataFrame({
        'relato': texts_test.values,
        'true': y_arr,
        'bow_pred': bow_arr,
        'deep_pred': deep_arr,
    })

    # tok2vec acertou, BOW errou (ganhos da profundidade)
    deep_wins = df_compare[disagree & (deep_arr == y_arr)]
    bow_wins  = df_compare[disagree & (bow_arr == y_arr)]

    print(f'\ntok2vec acertou, BOW errou: {len(deep_wins)} casos')
    print(f'BOW acertou, tok2vec errou: {len(bow_wins)} casos')

    print('\n--- tok2vec corrigiu casos críticos perdidos pelo BOW ---')
    improved_critico = deep_wins[deep_wins['true'] == 'critico']
    for _, row in improved_critico.head(3).iterrows():
        print(f"  true=crítico | bow={row['bow_pred']} | deep=crítico")
        print(f"  '{row['relato'][:180]}'")
        print()

## 7. Hibridização — ML Clássico + spaCy tok2vec

O `HybridClassifier` combina o melhor modelo clássico (NB03) com o spaCy.  
Testamos as três estratégias de fusão com o **modelo profundo**.

### Por que hibridizar?

| Componente | Ponto forte | Ponto fraco |
|-----------|-------------|-------------|
| ML Clássico (LogReg+TF-IDF) | Features estruturadas (área, turno) | Sem contexto de palavras |
| spaCy tok2vec | Contexto linguístico | Sem features numéricas/categóricas |
| **Híbrido** | **Ambos** | Meta-parâmetros a tunar |

In [ ]:
from src.features.build import prepare_dataframe

best_ml_path = ROOT / 'models/classic/best_model.joblib'

if not best_ml_path.exists():
    print('⚠️  Execute dvc repro train_classic para gerar o modelo clássico.')
    best_ml = None
else:
    best_ml = joblib.load(best_ml_path)
    print(f'Modelo clássico: {type(best_ml.named_steps["classifier"]).__name__}')

    X_train = prepare_dataframe(train, params)
    X_test_df = prepare_dataframe(test, params)

In [ ]:
if best_ml is not None:
    hybrid = HybridClassifier(params, best_ml, deep_trainer)

    # Treina meta-modelo do stack (usa dados de treino)
    hybrid.fit_stack(X_train, texts_train.values, y_train.values)

    # Avalia as três estratégias
    hybrid_results = {}
    for strategy in ['override', 'weighted', 'stack']:
        h_preds = hybrid.predict(X_test_df, texts_test.values, strategy=strategy)
        h_metrics = classification_metrics(y_test.values, h_preds)
        h_eace = eace_from_predictions(y_test.values, h_preds, params['eace'], CLASS_ORDER)
        h_metrics['eace_brl'] = h_eace
        hybrid_results[f'hybrid_{strategy}'] = h_metrics
        print(f"hybrid_{strategy:8s} | recall={h_metrics['recall_critico']:.4f} "
              f"| f1_macro={h_metrics['f1_macro']:.4f} | EACE=R${h_eace:,.0f}")

In [ ]:
if best_ml is not None:
    # Tabela completa: ML solo + spaCy solo + híbridos
    all_results = {}
    all_results['spacy_tok2vec'] = deep_metrics
    if bow_trainer is not None:
        all_results['spacy_bow'] = bow_metrics
    all_results.update(hybrid_results)

    table_all = compare_models_table(all_results)
    display(table_all.style
        .highlight_max(subset=['recall_critico', 'f1_macro'], color='#c8e6c9')
        .highlight_min(subset=['eace_brl'], color='#c8e6c9')
        .format({'eace_brl': 'R${:,.0f}'}))

In [ ]:
if best_ml is not None:
    fig = plot_metrics_comparison(all_results, metrics_to_plot=['recall_critico', 'f1_macro', 'accuracy'])
    plt.show()

## 8. Report por classe — melhor configuração híbrida

In [ ]:
if best_ml is not None and 'hybrid_results' in dir():
    # Escolhe estratégia com maior recall_critico
    best_strategy = max(hybrid_results, key=lambda k: hybrid_results[k]['recall_critico'])
    best_h_preds = hybrid.predict(X_test_df, texts_test.values, strategy=best_strategy.replace('hybrid_', ''))

    print(f'Melhor estratégia híbrida: {best_strategy}')
    report = per_class_report(y_test.values, best_h_preds, CLASS_ORDER)
    display(report.style
        .highlight_max(subset=['recall', 'f1'], color='#c8e6c9')
        .highlight_min(subset=['recall', 'f1'], color='#ffcdd2'))

## 9. SHAP no modelo tok2vec — explicabilidade via Kernel SHAP

O tok2vec não é linear, então não podemos usar `LinearExplainer`.  
Usamos `KernelExplainer` com uma amostra representativa como background.

> **Custo:** KernelSHAP é mais lento que LinearSHAP — usamos uma amostra pequena de exemplos.

In [ ]:
import shap

# Wrapper: SHAP precisa de uma função f(X) → probabilidades numéricas
def predict_proba_critico(texts_list):
    """Retorna array (n,) com probabilidade da classe crítico."""
    probas = deep_trainer.predict_proba(list(texts_list))
    from src.models.spacy_model import LABEL_MAP
    return probas[LABEL_MAP['critico']]

# Background: amostra estratificada de 50 exemplos de treino
rng = np.random.default_rng(params['base']['random_seed'])
background_idx = rng.choice(len(texts_train), size=50, replace=False)
background_texts = texts_train.values[background_idx]

# Explainer: usa KernelExplainer (model-agnostic)
# Cuidado: lento para muitos exemplos — use sample pequeno
print('Inicializando KernelExplainer (pode levar ~1-2 min)...')
kernel_explainer = shap.KernelExplainer(
    predict_proba_critico,
    background_texts,
    link='logit',
)
print('Explainer pronto.')

In [ ]:
# Seleciona 5 exemplos interessantes: mistura de críticos e não-críticos
critico_idx_test = np.where(np.array(y_test) == 'critico')[0][:3]
nao_critico_idx  = np.where(np.array(deep_preds) == 'critico')[0]
fp_idx = nao_critico_idx[np.array(y_test)[nao_critico_idx] != 'critico'][:2]  # falsos positivos

sample_idx = np.concatenate([critico_idx_test, fp_idx])
sample_texts = texts_test.values[sample_idx]
sample_labels = np.array(y_test)[sample_idx]

print(f'Calculando SHAP para {len(sample_texts)} exemplos...')
shap_kernel_values = kernel_explainer.shap_values(sample_texts, nsamples=100)
print('SHAP calculado.')

for i, (text, label, shap_vals) in enumerate(zip(sample_texts, sample_labels, shap_kernel_values)):
    pred = deep_preds[sample_idx[i]] if sample_idx[i] < len(deep_preds) else '?'
    print(f'\n--- Exemplo {i+1} (true={label}, pred={pred}) ---')
    print(f'{text[:150]}...' if len(text) > 150 else text)
    print(f'SHAP contribution para crítico: {shap_vals.sum():+.4f}')
    print(f'Base value: {kernel_explainer.expected_value:.4f}')

## 10. Demo Gradio — aplicação interativa

Gradio transforma o modelo em uma aplicação web com 20 linhas de código.  
A demo mostra:
1. **Entrada** de texto livre em português
2. **Predição** com probabilidade por classe (barra de cores)
3. **Explicabilidade** — top tokens que influenciaram a decisão

> Execute a célula abaixo — o Gradio abre um link público temporário.  
> Em produção, este bloco viraria um microserviço FastAPI + React.

In [ ]:
try:
    import gradio as gr
    _has_gradio = True
except ImportError:
    print('Gradio não instalado. Execute: pip install gradio')
    _has_gradio = False

In [ ]:
if _has_gradio:
    from src.models.spacy_model import LABEL_MAP, CLASS_ORDER as SP_CLASS_ORDER

    CLASS_COLORS = {
        'baixo':   '#4CAF50',
        'medio':   '#FF9800',
        'alto':    '#F44336',
        'critico': '#B71C1C',
    }

    def _format_probas(probas_dict: dict) -> dict:
        """Converte dict label→array para {classe: float} legível."""
        return {
            cls: float(probas_dict[LABEL_MAP[cls]][0])
            for cls in CLASS_ORDER
        }

    def predict_and_explain(relato: str):
        if not relato.strip():
            return 'Digite um relato.', {}, ''

        # Predição
        texts = [relato]
        probas = deep_trainer.predict_proba(texts)
        formatted = _format_probas(probas)
        pred_class = max(formatted, key=formatted.get)
        confidence = formatted[pred_class]

        result_text = (
            f'Classificação: **{pred_class.upper()}** '
            f'(confiança: {confidence:.1%})'
        )

        # Explicabilidade: tokens com maior score de risco crítico
        # Aproximação rápida via predict_proba por token removido (ablation)
        tokens = relato.lower().split()
        critico_base = formatted.get('critico', 0.0)
        token_impacts = []
        for i, tok in enumerate(tokens):
            masked = ' '.join(t for j, t in enumerate(tokens) if j != i)
            p_masked = deep_trainer.predict_proba([masked])
            critico_masked = float(p_masked[LABEL_MAP['critico']][0])
            impact = critico_base - critico_masked  # positivo = token importante
            token_impacts.append((tok, impact))

        token_impacts.sort(key=lambda x: x[1], reverse=True)

        top_positive = [f"{t} (+{v:.3f})" for t, v in token_impacts if v > 0.005][:5]
        top_negative = [f"{t} ({v:+.3f})" for t, v in token_impacts if v < -0.005][:3]

        explain_lines = []
        if top_positive:
            explain_lines.append(f"Tokens que AUMENTAM risco critico: {', '.join(top_positive)}")
        if top_negative:
            explain_lines.append(f"Tokens que REDUZEM risco critico: {', '.join(top_negative)}")
        if not explain_lines:
            explain_lines = ['Nenhum token com impacto significativo detectado.']

        explanation = '\n'.join(explain_lines)

        return result_text, formatted, explanation

    demo = gr.Interface(
        fn=predict_and_explain,
        inputs=gr.Textbox(
            label='Relato de incidente (português)',
            placeholder='Ex: Operador identificou chama aberta próxima ao manifold de gás durante inspeção de rotina.',
            lines=4,
        ),
        outputs=[
            gr.Markdown(label='Resultado'),
            gr.Label(label='Probabilidade por classe', num_top_classes=4),
            gr.Textbox(label='Explicabilidade (ablation por token)', lines=4),
        ],
        title='Classificador de Risco — FPSO Safety (spaCy tok2vec)',
        description=(
            'Classifica relatos de incidentes em FPSOs nas classes: '
            'baixo / médio / alto / crítico.\n'
            'Explicabilidade por ablação: remove cada token e mede '
            'o impacto na probabilidade da classe crítico.'
        ),
        examples=[
            ['Operador identificou chama aberta próxima ao manifold de gás durante inspeção de rotina.'],
            ['Vazamento de óleo detectado na linha de produção, volume estimado em 5 litros.'],
            ['Substituição de válvula realizada sem intercorrências durante parada programada.'],
            ['Trabalhador escorregou e machucou o joelho ao descer escada da plataforma de produção.'],
        ],
        theme=gr.themes.Soft(),
    )

    demo.launch(share=True, quiet=True)
    print('\nDemo Gradio iniciada — acesse o link acima.')
    print('Para encerrar: demo.close() ou reinicie o kernel.')

## 11. Síntese: o que aprendemos no Tier 2?

| Dimensão | BOW | tok2vec/ensemble | Híbrido (melhor) |
|----------|-----|-----------------|------------------|
| Recall crítico | ver tabela | ver tabela | ver tabela |
| Capta contexto | Não | Parcialmente | Parcialmente |
| Capta features estruturais | Não | Não | **Sim** |
| Treino (segundos) | ~30s | ~2-5min | +~1min |
| Explicabilidade | Implícita (scores) | Ablação/SHAP | Ablação/SHAP |
| Custo inferência | Baixo | Médio | Médio |

### Quando usar o híbrido vs. spaCy solo?

- **spaCy solo** → quando só há texto disponível (sem metadados de área/turno)
- **Híbrido override** → quando recall é crítico e falsos positivos são aceitáveis
- **Híbrido stack** → quando a distribuição de treino e produção são estáveis

---

## 12. Próximos passos

| Tier | Notebook | O que muda |
|------|----------|------------|
| Classic + SHAP | NB03 | TF-IDF + LogReg — baseline interpretável |
| spaCy BOW + tok2vec (este) | NB04 | NLP nativo + hibridização |
| **LLM** | **NB05** | **Zero-shot e few-shot com Claude — sem treino** |
| Benchmark | NB06 | Comparação cross-tier + Gradio final |

**Pergunta para o NB05:** um LLM sem qualquer fine-tuning supera o recall@crítico do melhor modelo treinado?